# SurfaceTarget-CRC — phase 03 — patch 3 2

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 3 — PATCH 3.2
#
# UNIPROT PROTEIN TOPOLOGY AND EXTRACELLULAR-DOMAIN AUDIT
#
# Objectives:
# 1. Load HPA-integrated candidate table
# 2. Map Ensembl gene IDs to UniProtKB entries
# 3. Prefer uniquely mapped reviewed human entries
# 4. Preserve ambiguous and missing mappings
# 5. Extract subcellular localization annotations
# 6. Identify transmembrane and GPI-anchor evidence
# 7. Extract annotated extracellular topological regions
# 8. Integrate results with RNA, CPTAC and HPA evidence
# 9. Review leading candidates
# 10. Save raw source data, tables and provenance
#
# IMPORTANT:
# - No gene-symbol-only mapping
# - No arbitrary choice among reviewed mappings
# - No automatic candidate exclusions
# - No final therapeutic ranking
# - No assumption that membrane association proves
#   extracellular antibody accessibility
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
import numpy as np
import requests

import hashlib
import json
import gzip
import re
import time


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/uniprot"
RESULTS = ROOT / "results/tables"

for directory in [META, RAW, RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    RESULTS /
    "surfacetarget_hpa_annotation_audit.csv"
)

RAW_MAPPING_PATH = (
    RAW /
    "uniprot_ensembl_mapping_results.json.gz"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 3 — PATCH 3.2")
print("UNIPROT PROTEIN TOPOLOGY AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. LOAD CANDIDATE EVIDENCE
# ------------------------------------------------------------

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Patch 3.1 HPA annotation table missing."
    )

candidates = pd.read_csv(
    INPUT_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)

required_columns = {
    "gene_id",
    "gene_name",
    "ensembl_stable_id",
    "protein_enriched"
}

if not required_columns.issubset(candidates.columns):
    raise RuntimeError(
        "Required candidate columns missing."
    )

if len(candidates) != 948:
    raise RuntimeError(
        "Expected 948 candidate genes."
    )

if candidates["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate stable Ensembl gene IDs."
    )

candidate_ids = (
    candidates["ensembl_stable_id"]
    .astype(str)
    .tolist()
)

candidate_id_set = set(candidate_ids)

input_fingerprint = hashlib.sha256(
    "\n".join(sorted(candidate_ids)).encode("utf-8")
).hexdigest()

print("\nCandidate evidence loaded.")

print("Candidate genes:", len(candidates))

print("Stable Ensembl identifiers: VERIFIED")


# ------------------------------------------------------------
# 3. UNIPROT API CONFIGURATION
# ------------------------------------------------------------

API_BASE = "https://rest.uniprot.org"

session = requests.Session()

retry_policy = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[
        429, 500, 502, 503, 504
    ],
    allowed_methods=["GET", "POST"]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry_policy)
)

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0",
    "Accept": "application/json"
})


def get_json(url, **kwargs):

    response = session.get(
        url,
        timeout=(30, 120),
        **kwargs
    )

    response.raise_for_status()

    return response.json()


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. ACQUIRE UNIPROT IDENTIFIER MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("UNIPROT IDENTIFIER MAPPING")
print("-" * 65)

mapping_data = None
mapping_job_id = None

# Reuse previously completed API results
# only when the requested Ensembl IDs match.

if RAW_MAPPING_PATH.exists():

    with gzip.open(
        RAW_MAPPING_PATH,
        "rt",
        encoding="utf-8"
    ) as handle:

        cached = json.load(handle)

    if (
        cached.get("input_fingerprint")
        == input_fingerprint
        and "results" in cached
    ):

        mapping_data = cached

        mapping_job_id = cached.get("job_id")

        print("\nExisting matching UniProt results reused.")


if mapping_data is None:

    print("\nSubmitting Ensembl-to-UniProtKB mapping job.")

    response = session.post(
        f"{API_BASE}/idmapping/run",
        data={
            "from": "Ensembl",
            "to": "UniProtKB",
            "ids": ",".join(candidate_ids)
        },
        timeout=(30, 120)
    )

    response.raise_for_status()

    submission = response.json()

    mapping_job_id = submission.get("jobId")

    if not mapping_job_id:
        raise RuntimeError(
            "UniProt mapping job ID not returned."
        )

    print("Mapping job submitted.")

    print("\nChecking mapping status...")

    completed = False

    for attempt in range(120):

        status_response = get_json(
            f"{API_BASE}/idmapping/status/"
            f"{mapping_job_id}"
        )

        job_status = status_response.get("jobStatus")

        if job_status in {"NEW", "RUNNING"}:

            if attempt % 10 == 0:
                print("Mapping job processing...")

            time.sleep(3)
            continue

        if job_status in {"ERROR", "FAILED"}:
            raise RuntimeError(
                f"UniProt mapping failed: {status_response}"
            )

        # A completed mapping response may contain
        # results or failedIds without jobStatus.

        if (
            job_status == "FINISHED"
            or "results" in status_response
            or "failedIds" in status_response
        ):

            completed = True
            break

        raise RuntimeError(
            f"Unexpected mapping status: {status_response}"
        )

    if not completed:
        raise RuntimeError(
            "UniProt mapping job did not complete "
            "during this execution."
        )

    # Retrieve the official result URL.

    details = get_json(
        f"{API_BASE}/idmapping/details/"
        f"{mapping_job_id}"
    )

    results_url = details.get("redirectURL")

    if not results_url:
        raise RuntimeError(
            "UniProt results URL missing."
        )

    separator = "&" if "?" in results_url else "?"

    next_url = (
        results_url
        + separator
        + "format=json&size=500"
    )

    all_results = []
    failed_ids = set()
    page_count = 0

    # Retrieve all pages, not only the first 500
    # mappings returned by the API.

    while next_url:

        page_count += 1

        if page_count > 100:
            raise RuntimeError(
                "Unexpectedly many UniProt result pages."
            )

        response = session.get(
            next_url,
            timeout=(30, 120)
        )

        response.raise_for_status()

        payload = response.json()

        all_results.extend(
            payload.get("results", [])
        )

        for failed in payload.get("failedIds", []):

            if isinstance(failed, dict):
                failed = failed.get("from", "")

            failed_ids.add(str(failed))

        link_header = response.headers.get(
            "Link", ""
        )

        next_match = re.search(
            r'<([^>]+)>;\s*rel="next"',
            link_header
        )

        next_url = (
            next_match.group(1)
            if next_match
            else None
        )

        print(
            f"Retrieved page {page_count}: "
            f"{len(all_results)} mapping records"
        )

    mapping_data = {
        "source": "UniProt REST API",
        "from_database": "Ensembl",
        "to_database": "UniProtKB",
        "job_id": mapping_job_id,
        "retrieved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "input_fingerprint": input_fingerprint,
        "input_count": len(candidate_ids),
        "results": all_results,
        "failedIds": sorted(failed_ids)
    }

    # Preserve complete API mapping results,
    # including alternative UniProt entries.

    with gzip.open(
        RAW_MAPPING_PATH,
        "wt",
        encoding="utf-8"
    ) as handle:

        json.dump(
            mapping_data,
            handle,
            separators=(",", ":")
        )

    print("\nRaw UniProt mapping results saved.")


print("\nMapping records:",
      len(mapping_data["results"]))

print("Unmapped identifiers reported:",
      len(mapping_data.get("failedIds", [])))


# ------------------------------------------------------------
# 5. VALIDATE MAPPING RESPONSES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("UNIPROT MAPPING VALIDATION")
print("-" * 65)

mapping_by_gene = defaultdict(dict)

unexpected_source_ids = set()

for record in mapping_data["results"]:

    ensembl_id = str(
        record.get("from", "")
    ).strip()

    if ensembl_id not in candidate_id_set:
        unexpected_source_ids.add(ensembl_id)
        continue

    entry = record.get("to")

    if not isinstance(entry, dict):
        raise RuntimeError(
            "UniProt mapping response lacks "
            "a structured protein entry."
        )

    accession = entry.get(
        "primaryAccession"
    )

    if not accession:
        raise RuntimeError(
            "Mapped UniProt entry missing accession."
        )

    organism = entry.get(
        "organism", {}
    )

    taxon_id = organism.get(
        "taxonId"
    )

    if str(taxon_id) != "9606":
        # Do not accept nonhuman entries.
        continue

    mapping_by_gene[ensembl_id][accession] = entry


if unexpected_source_ids:
    raise RuntimeError(
        "UniProt returned unexpected input IDs: "
        f"{sorted(unexpected_source_ids)[:10]}"
    )

print("\nHuman UniProt mappings parsed.")

print(
    "Candidate genes with any human mapping:",
    len(mapping_by_gene)
)


# ------------------------------------------------------------
# 6. SELECT UNAMBIGUOUS REVIEWED ENTRIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWED ENTRY SELECTION")
print("-" * 65)

selection_records = []

selected_entries = {}


def is_reviewed(entry):

    entry_type = str(
        entry.get("entryType", "")
    ).lower()

    return (
        "reviewed" in entry_type
        and "unreviewed" not in entry_type
    )


for gene_id in candidate_ids:

    entries = list(
        mapping_by_gene.get(
            gene_id, {}
        ).values()
    )

    reviewed = [
        entry
        for entry in entries
        if is_reviewed(entry)
    ]

    if len(reviewed) == 1:

        status = "Unique reviewed mapping"

        selected_entries[gene_id] = reviewed[0]

    elif len(reviewed) > 1:

        status = "Ambiguous reviewed mappings"

    elif entries:

        status = "Unreviewed mappings only"

    else:

        status = "No human UniProt mapping"

    selection_records.append({

        "ensembl_stable_id": gene_id,

        "total_human_uniprot_entries":
            len(entries),

        "reviewed_entries":
            len(reviewed),

        "mapping_status":
            status,

        "reviewed_accessions":
            "; ".join(
                sorted(
                    entry["primaryAccession"]
                    for entry in reviewed
                )
            )
    })


selection_table = pd.DataFrame(
    selection_records
)

print("\nMapping status distribution:")

print(
    selection_table[
        "mapping_status"
    ].value_counts().to_string()
)

print(
    "\nUniquely mapped reviewed proteins:",
    len(selected_entries)
)

print(
    "Ambiguous mappings are retained "
    "for review, not automatically resolved."
)


# ------------------------------------------------------------
# 7. EXTRACT UNIPROT TOPOLOGY ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING PROTEIN TOPOLOGY")
print("-" * 65)


def extract_gene_symbol(entry):

    for gene in entry.get("genes", []):

        name = gene.get(
            "geneName", {}
        )

        if isinstance(name, dict):

            value = name.get("value")

            if value:
                return str(value)

    return ""


def extract_locations(entry):

    locations = []

    for comment in entry.get("comments", []):

        if comment.get("commentType") != (
            "SUBCELLULAR LOCATION"
        ):
            continue

        for item in comment.get(
            "subcellularLocations", []
        ):

            location = item.get(
                "location", {}
            )

            if isinstance(location, dict):

                value = location.get(
                    "value", ""
                )

                if value:
                    locations.append(str(value))

    return sorted(set(locations))


def feature_length(feature):

    location = feature.get(
        "location", {}
    )

    start = location.get(
        "start", {}
    )

    end = location.get(
        "end", {}
    )

    if not isinstance(start, dict):
        return None

    if not isinstance(end, dict):
        return None

    first = start.get("value")
    last = end.get("value")

    if first is None or last is None:
        return None

    try:

        first = int(first)
        last = int(last)

        if last < first:
            return None

        return last - first + 1

    except (TypeError, ValueError):

        return None


annotation_records = []


for ensembl_id, entry in selected_entries.items():

    accession = entry["primaryAccession"]

    features = entry.get(
        "features", []
    )

    locations = extract_locations(
        entry
    )

    location_text = "; ".join(
        locations
    )

    # Identify reported localization.

    cell_membrane = any(
        re.search(
            r"\b(cell|plasma) membrane\b",
            location,
            flags=re.IGNORECASE
        )
        for location in locations
    )

    secreted = any(
        "secreted" in location.lower()
        or "extracellular space" in location.lower()
        for location in locations
    )

    # Examine sequence features.

    transmembrane_features = []

    extracellular_features = []

    signal_features = []

    lipidation_features = []

    for feature in features:

        feature_type = feature.get(
            "type", ""
        )

        description = str(
            feature.get(
                "description", ""
            )
        )

        if feature_type == "Transmembrane":

            transmembrane_features.append(
                feature
            )

        elif feature_type == "Topological domain":

            if re.search(
                r"\bextracellular\b",
                description,
                flags=re.IGNORECASE
            ):

                extracellular_features.append(
                    feature
                )

        elif feature_type == "Signal":

            signal_features.append(
                feature
            )

        elif feature_type == "Lipidation":

            lipidation_features.append(
                feature
            )

    # GPI-anchor annotations may appear
    # in localization comments or features.

    gpi_feature = any(
        "gpi" in str(
            feature.get(
                "description", ""
            )
        ).lower()
        for feature in lipidation_features
    )

    gpi_location = any(
        "gpi" in location.lower()
        for location in locations
    )

    gpi_anchor = (
        gpi_feature or gpi_location
    )

    # Count only explicitly annotated
    # extracellular topological regions.

    extracellular_lengths = [
        feature_length(feature)
        for feature in extracellular_features
    ]

    extracellular_lengths = [
        length
        for length in extracellular_lengths
        if length is not None
    ]

    total_extracellular_aa = (
        sum(extracellular_lengths)
        if extracellular_lengths
        else np.nan
    )

    transmembrane_count = len(
        transmembrane_features
    )

    extracellular_count = len(
        extracellular_features
    )

    # Descriptive evidence category.
    #
    # This is NOT an antibody-accessibility
    # prediction or target ranking.

    if (
        cell_membrane
        and (
            transmembrane_count > 0
            or gpi_anchor
        )
        and extracellular_count > 0
    ):

        topology_category = (
            "Cell membrane with annotated "
            "extracellular region"
        )

    elif (
        cell_membrane
        and (
            transmembrane_count > 0
            or gpi_anchor
        )
    ):

        topology_category = (
            "Cell membrane; extracellular "
            "topology not explicitly annotated"
        )

    elif cell_membrane:

        topology_category = (
            "Cell-membrane localization reported; "
            "attachment mechanism unresolved"
        )

    elif secreted:

        topology_category = (
            "Secreted or extracellular; "
            "not established as membrane-bound"
        )

    elif transmembrane_count > 0:

        topology_category = (
            "Transmembrane protein; "
            "cell-surface localization unresolved"
        )

    else:

        topology_category = (
            "No qualifying cell-surface "
            "topology evidence in extracted annotations"
        )

    sequence = entry.get(
        "sequence", {}
    )

    annotation_records.append({

        "ensembl_stable_id": ensembl_id,

        "uniprot_accession": accession,

        "uniprot_entry_name":
            entry.get("uniProtkbId", ""),

        "uniprot_gene_symbol":
            extract_gene_symbol(entry),

        "uniprot_entry_type":
            entry.get("entryType", ""),

        "uniprot_sequence_length":
            sequence.get("length"),

        "uniprot_subcellular_locations":
            location_text,

        "uniprot_cell_membrane":
            cell_membrane,

        "uniprot_secreted":
            secreted,

        "uniprot_transmembrane_count":
            transmembrane_count,

        "uniprot_signal_peptide":
            len(signal_features) > 0,

        "uniprot_gpi_anchor":
            gpi_anchor,

        "uniprot_extracellular_domain_count":
            extracellular_count,

        "uniprot_annotated_extracellular_aa":
            total_extracellular_aa,

        "uniprot_topology_category":
            topology_category,

        "uniprot_entry_url":
            (
                "https://www.uniprot.org/"
                "uniprotkb/"
                f"{accession}/entry"
            )
    })


annotations = pd.DataFrame(
    annotation_records
)


if annotations["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate selected protein annotations."
    )


print("\nProtein topology extraction: COMPLETE")

print("Reviewed protein annotations:", len(annotations))


# ------------------------------------------------------------
# 8. INTEGRATE UNIPROT WITH EXISTING EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING PROTEIN TOPOLOGY EVIDENCE")
print("-" * 65)


integrated = candidates.merge(
    selection_table,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)

integrated = integrated.merge(
    annotations,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)


if len(integrated) != 948:
    raise RuntimeError(
        "Candidate count changed during integration."
    )


integrated["uniprot_reviewed_available"] = (
    integrated["uniprot_accession"].notna()
)


# Gene-symbol concordance is recorded for review.
# We do not discard mappings based solely
# on gene-symbol changes or aliases.

integrated["uniprot_gene_symbol_match"] = (
    integrated["gene_name"]
    .astype(str)
    .str.upper()
    .eq(
        integrated["uniprot_gene_symbol"]
        .fillna("")
        .astype(str)
        .str.upper()
    )
)

integrated.loc[
    ~integrated["uniprot_reviewed_available"],
    "uniprot_gene_symbol_match"
] = False


print("\nCandidate integration: PASSED")

print("Total genes:", len(integrated))

print(
    "With uniquely mapped reviewed UniProt entries:",
    int(
        integrated[
            "uniprot_reviewed_available"
        ].sum()
    )
)

print(
    "With UniProt cell-membrane localization:",
    int(
        integrated[
            "uniprot_cell_membrane"
        ].fillna(False).sum()
    )
)

print(
    "With annotated extracellular regions:",
    int(
        integrated[
            "uniprot_extracellular_domain_count"
        ].fillna(0).gt(0).sum()
    )
)

print(
    "With GPI-anchor annotations:",
    int(
        integrated[
            "uniprot_gpi_anchor"
        ].fillna(False).sum()
    )
)

print("\nTopology categories:")

print(
    integrated[
        "uniprot_topology_category"
    ]
    .fillna("UniProt mapping unresolved")
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 9. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE TOPOLOGY REVIEW")
print("-" * 65)


leading_genes = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


leading = integrated[
    integrated["gene_name"].isin(
        leading_genes
    )
].copy()


order_lookup = {
    gene: index
    for index, gene in enumerate(
        leading_genes
    )
}

leading["display_order"] = (
    leading["gene_name"].map(
        order_lookup
    )
)

leading = leading.sort_values(
    "display_order"
)


display_columns = [
    "gene_name",
    "mapping_status",
    "uniprot_accession",
    "uniprot_subcellular_locations",
    "uniprot_transmembrane_count",
    "uniprot_gpi_anchor",
    "uniprot_extracellular_domain_count",
    "uniprot_annotated_extracellular_aa"
]


print("\n")

print(
    leading[display_columns]
    .to_string(
        index=False
    )
)


print("\nIMPORTANT:")

print(
    "Annotated extracellular regions support "
    "topological plausibility, not demonstrated "
    "therapeutic antibody accessibility."
)

print(
    "Missing topology annotations must not "
    "be interpreted as evidence of absence."
)


# ------------------------------------------------------------
# 10. REVIEW CROSS-OMICS-SUPPORTED PROTEINS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS PROTEIN TOPOLOGY")
print("-" * 65)


cross_omics_mask = (
    integrated["protein_enriched"]
    .fillna(False)
    .astype(str)
    .str.lower()
    .eq("true")
)


cross_omics = integrated[
    cross_omics_mask
].copy()


if len(cross_omics) != 20:
    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


print("\nCross-omics-supported proteins:", len(cross_omics))

print("\nTopology categories among those proteins:")

print(
    cross_omics[
        "uniprot_topology_category"
    ]
    .fillna("UniProt mapping unresolved")
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 11. SAVE ANNOTATION AND MAPPING TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING UNIPROT ANNOTATION TABLES")
print("-" * 65)


selection_table.to_csv(
    RESULTS /
    "uniprot_ensembl_mapping_audit.csv",
    index=False
)

annotations.to_csv(
    RESULTS /
    "uniprot_candidate_topology_annotations.csv",
    index=False
)

integrated.to_csv(
    RESULTS /
    "surfacetarget_uniprot_topology_integration.csv",
    index=False
)

leading.to_csv(
    RESULTS /
    "surfacetarget_uniprot_leading_gene_review.csv",
    index=False
)

cross_omics.to_csv(
    RESULTS /
    "surfacetarget_crossomics_topology_review.csv",
    index=False
)


print("\nAnnotation tables saved successfully.")


# ------------------------------------------------------------
# 12. SAVE PROVENANCE
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "3.2",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "data_source": "UniProtKB REST API",

    "api_base": API_BASE,

    "mapping_from": "Ensembl",

    "mapping_to": "UniProtKB",

    "mapping_job_id": mapping_job_id,

    "mapping_input_fingerprint":
        input_fingerprint,

    "mapping_input_genes":
        len(candidate_ids),

    "mapping_result_records":
        len(mapping_data["results"]),

    "raw_mapping_path":
        str(RAW_MAPPING_PATH),

    "raw_mapping_sha256":
        sha256_file(RAW_MAPPING_PATH),

    "reviewed_mapping_policy": (
        "Accept exactly one reviewed human "
        "UniProtKB accession per Ensembl gene; "
        "retain ambiguous mappings for review"
    ),

    "uniquely_mapped_reviewed_genes":
        len(selected_entries),

    "cross_omics_supported_genes":
        len(cross_omics),

    "annotation_types": [
        "SUBCELLULAR LOCATION",
        "Transmembrane",
        "Topological domain",
        "Signal",
        "Lipidation"
    ],

    "extracellular_region_rule": (
        "Explicit extracellular description "
        "within UniProt Topological domain features"
    ),

    "candidate_exclusions": 0,

    "antibody_accessibility_verified": False,

    "target_ranking_performed": False,

    "limitations": [

        "UniProt sequence topology does not "
        "prove antibody binding.",

        "Different protein isoforms may have "
        "different topology.",

        "Only uniquely mapped reviewed human "
        "entries receive topology annotations.",

        "Ambiguous mappings remain unresolved.",

        "Missing UniProt annotations are "
        "not negative biological evidence.",

        "GPI attachment and transmembrane "
        "anchoring are distinct mechanisms.",

        "Secreted proteins are not necessarily "
        "cell-surface anchored.",

        "Cell-membrane annotations may reflect "
        "non-tumor biological contexts.",

        "CPTAC protein enrichment does not "
        "establish malignant-cell expression.",

        "Normal-tissue safety is not established.",

        "No therapeutic targets have "
        "been validated."
    ]
}


with open(
    META /
    "uniprot_topology_audit_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 3.2 SUMMARY")
print("=" * 65)


print("\nTranscriptomic screening genes:", len(integrated))

print("Cross-omics-supported genes:", len(cross_omics))

print(
    "Uniquely mapped reviewed proteins:",
    len(selected_entries)
)

print(
    "Cell-membrane-localized proteins:",
    int(
        integrated[
            "uniprot_cell_membrane"
        ].fillna(False).sum()
    )
)

print(
    "Proteins with extracellular-domain annotations:",
    int(
        integrated[
            "uniprot_extracellular_domain_count"
        ].fillna(0).gt(0).sum()
    )
)


print("\nFiles saved:")

print("1. uniprot_ensembl_mapping_audit.csv")

print("2. uniprot_candidate_topology_annotations.csv")

print("3. surfacetarget_uniprot_topology_integration.csv")

print("4. surfacetarget_uniprot_leading_gene_review.csv")

print("5. surfacetarget_crossomics_topology_review.csv")

print("6. uniprot_topology_audit_provenance.json")

print("7. uniprot_ensembl_mapping_results.json.gz")


print("\nNo genes automatically excluded.")

print("No therapeutic target scores assigned.")

print("No TCGA, CPTAC or HPA results overwritten.")


print("\nPATCH 3.2 COMPLETE")

print("=" * 65)